# PayGuard — Model 4: Isolation Forest

**Unsupervised** anomaly detection — builds many random trees that isolate each data point via random feature splits. Anomalies get isolated in fewer splits because they differ significantly from the bulk of the data, so points with shorter average path lengths across the trees are scored as more anomalous.

**What's different here vs. the three supervised models:**
- Training uses **only `X_train_scaled`** — `y_train` is never passed to `.fit()`. The model has no idea which rows are fraud; it just learns what "normal" looks like.
- There's no SMOTE/undersampling/class-weighting here — those are resampling techniques for *labeled* data, and this model doesn't use labels at all.
- Instead, the one knob we *do* have is `contamination` — Isolation Forest's assumption about what fraction of the data is anomalous. We compare a few settings for this, which plays the same "compare preprocessing/tuning choices" role that SMOTE/undersampling/class-weighting played for the supervised models.

In [1]:
import joblib
import os
import numpy as np
from sklearn.ensemble import IsolationForest
from evaluation import evaluate_model, print_comparison_table

bundle = joblib.load("../data/processed/train_test_variants.pkl")

X_train_scaled = bundle["X_train_scaled"]
y_train = bundle["y_train"]  # only used to inspect the true fraud rate, never passed to .fit()
X_test_scaled = bundle["X_test_scaled"]
y_test = bundle["y_test"]

true_fraud_rate = y_train.mean()
print(f"True training fraud rate (for reference only): {true_fraud_rate:.4f}")
print("Bundle loaded. Train shape:", X_train_scaled.shape, "| Test shape:", X_test_scaled.shape)

True training fraud rate (for reference only): 0.0028
Bundle loaded. Train shape: (64000, 21) | Test shape: (16000, 21)


## Helper: convert Isolation Forest's output into our standard fraud/not-fraud format

`IsolationForest.predict()` returns `-1` for an anomaly (outlier) and `1` for normal — the opposite convention from our `is_fraud` column (`1` = fraud). `decision_function()` gives higher values for "more normal," so we flip its sign to get an anomaly score where higher = more fraud-like, matching what `evaluate_model`'s `y_score` expects for ROC-AUC.

In [2]:
def run_isolation_forest(contamination, label):
    model = IsolationForest(
        n_estimators=200, contamination=contamination, random_state=42, n_jobs=-1,
    )
    model.fit(X_train_scaled)  # NOTE: no y_train passed — this is the unsupervised part

    raw_pred = model.predict(X_test_scaled)             # -1 = anomaly, 1 = normal
    y_pred = np.where(raw_pred == -1, 1, 0)              # convert to 1 = fraud, 0 = not fraud
    y_score = -model.decision_function(X_test_scaled)    # flip sign: higher = more anomalous

    result = evaluate_model(y_test, y_pred, "Isolation Forest", label, y_score)
    return model, result

## Variant 1: Contamination calibrated to the true training fraud rate

The most realistic setting — assumes we roughly know the historical fraud rate (a fair assumption for a bank that already logs confirmed fraud cases, even if sparse).

In [3]:
iso_calibrated, result_calibrated = run_isolation_forest(true_fraud_rate, "Calibrated to True Rate")


  Isolation Forest (Calibrated to True Rate)
Precision: 0.0222   Recall: 0.0222   F1: 0.0222

Confusion Matrix:
                 Predicted Not Fraud   Predicted Fraud
Actual Not Fraud           15911                  44
Actual Fraud                  44                   1

ROC-AUC: 0.8055

Full classification report:
              precision    recall  f1-score   support

   Not Fraud       1.00      1.00      1.00     15955
       Fraud       0.02      0.02      0.02        45

    accuracy                           0.99     16000
   macro avg       0.51      0.51      0.51     16000
weighted avg       0.99      0.99      0.99     16000



## Variant 2: `contamination="auto"`

scikit-learn's built-in heuristic, used when you *don't* know the true fraud rate in advance — the more realistic scenario for a brand-new fraud system with no historical labels yet.

In [4]:
iso_auto, result_auto = run_isolation_forest("auto", "Auto Contamination")


  Isolation Forest (Auto Contamination)
Precision: 0.0153   Recall: 0.5333   F1: 0.0298

Confusion Matrix:
                 Predicted Not Fraud   Predicted Fraud
Actual Not Fraud           14412                1543
Actual Fraud                  21                  24

ROC-AUC: 0.8055

Full classification report:
              precision    recall  f1-score   support

   Not Fraud       1.00      0.90      0.95     15955
       Fraud       0.02      0.53      0.03        45

    accuracy                           0.90     16000
   macro avg       0.51      0.72      0.49     16000
weighted avg       1.00      0.90      0.95     16000



## Variant 3: Wider net (2x the true rate)

A deliberately more conservative setting — assumes fraud might be under-reported historically (plausible, since not all fraud gets confirmed/labeled) and casts a wider net.

In [5]:
iso_wide, result_wide = run_isolation_forest(min(true_fraud_rate * 2, 0.5), "Wider Net (2x Rate)")


  Isolation Forest (Wider Net (2x Rate))
Precision: 0.0400   Recall: 0.0667   F1: 0.0500

Confusion Matrix:
                 Predicted Not Fraud   Predicted Fraud
Actual Not Fraud           15883                  72
Actual Fraud                  42                   3

ROC-AUC: 0.8055

Full classification report:
              precision    recall  f1-score   support

   Not Fraud       1.00      1.00      1.00     15955
       Fraud       0.04      0.07      0.05        45

    accuracy                           0.99     16000
   macro avg       0.52      0.53      0.52     16000
weighted avg       0.99      0.99      0.99     16000



## Compare the three contamination settings

In [6]:
iso_results = [result_calibrated, result_auto, result_wide]
iso_comparison_df = print_comparison_table(iso_results)


  MODEL COMPARISON (sorted by F1)
           model                 variant  precision  recall     f1  roc_auc  true_positives  false_positives  true_negatives  false_negatives
Isolation Forest     Wider Net (2x Rate)     0.0400  0.0667 0.0500   0.8055               3               72           15883               42
Isolation Forest      Auto Contamination     0.0153  0.5333 0.0298   0.8055              24             1543           14412               21
Isolation Forest Calibrated to True Rate     0.0222  0.0222 0.0222   0.8055               1               44           15911               44


## A fairer comparison: ROC-AUC

Because `contamination` directly controls how many points get labeled "anomaly" (it's basically setting the decision threshold), precision/recall/F1 are **threshold-dependent** here — a different contamination value just moves the cutoff, not the model's actual ability to separate fraud from non-fraud. **ROC-AUC is threshold-independent** and is the fairer number to compare against the supervised models, since it measures the quality of the anomaly *ranking* itself, before any cutoff is applied.

In [7]:
print("ROC-AUC is identical across all three variants above (same underlying model/scores,")
print("only the threshold used to convert scores into 0/1 predictions changes):")
print(iso_comparison_df[["variant", "roc_auc"]])

ROC-AUC is identical across all three variants above (same underlying model/scores,
only the threshold used to convert scores into 0/1 predictions changes):
                   variant  roc_auc
0      Wider Net (2x Rate)   0.8055
1       Auto Contamination   0.8055
2  Calibrated to True Rate   0.8055


## Save the best variant's model + results

In [8]:
os.makedirs("../models", exist_ok=True)

best_variant = iso_comparison_df.iloc[0]
print(f"Best variant for Isolation Forest: {best_variant['variant']} (F1={best_variant['f1']}, recall={best_variant['recall']})")

variant_to_model = {
    "Calibrated to True Rate": iso_calibrated,
    "Auto Contamination": iso_auto,
    "Wider Net (2x Rate)": iso_wide,
}
best_model = variant_to_model[best_variant["variant"]]

joblib.dump(best_model, "../models/isolation_forest.pkl")
iso_comparison_df.to_csv("../models/isolation_forest_results.csv", index=False)
print("Saved model to ../models/isolation_forest.pkl")
print("Saved results to ../models/isolation_forest_results.csv")

Best variant for Isolation Forest: Wider Net (2x Rate) (F1=0.05, recall=0.0667)
Saved model to ../models/isolation_forest.pkl
Saved results to ../models/isolation_forest_results.csv


## Next: `04e_model_one_class_svm.ipynb`

The second and final unsupervised model — same structure as this notebook, but using a completely different mechanism: a learned decision boundary around the "normal" region in feature space, rather than isolation-by-splitting.